# MiniGPT: 50M Parameter Training Run

This notebook will download, install, and train the MiniGPT model from scratch using a GPU. 
**Make sure to set your runtime to GPU (T4 is fine).**

In [ ]:
# 1. Clone the repository
!git clone https://github.com/mrshibly/MiniGPT-from-Scratch.git
%cd MiniGPT-from-Scratch
!pip install -r requirements.txt

## Step 1: Download & Preprocess Data
We will download ~500MB of the FineWeb-Edu dataset (much larger than the default 10MB) for this full training run.

In [ ]:
# Open download_fineweb.py and change limit to 500MB
!sed -i 's/max_bytes = 10 \* 1024 \* 1024/max_bytes = 500 \* 1024 \* 1024/' src/datasets/download_fineweb.py

# Run Data Pipeline
!python src/datasets/download_fineweb.py
!python src/datasets/clean_text.py
!python src/tokenizer/train_tokenizer.py
!python src/datasets/prepare_data.py

## Step 2: Configure the 50-Million Parameter Model

The default in `train.py` is `MiniGPTConfig.tiny()`. We want to scale this up to ~50 Million parameters.

In [ ]:
# Rewrite train.py configurations via sed commands
!sed -i 's/config = MiniGPTConfig.tiny()/config = MiniGPTConfig.standard()/' src/train/train.py
!sed -i 's/max_steps = 2000/max_steps = 10000/' src/train/train.py
!sed -i 's/max_lr = 6e-4/max_lr = 4e-4/' src/train/train.py
!sed -i 's/eval_interval = 250/eval_interval = 500/' src/train/train.py
!sed -i 's/batch_size = 16/batch_size = 8/' src/train/train.py

## Step 3: Train!
This will take a few hours. Checkpoints will automatically be saved.

In [ ]:
!python src/train/train.py

## Step 4: Evaluate Model
Once training finishes, calculate the model's perplexity.

In [ ]:
!python src/model/evaluate.py